# Tool choice


tool use는 Claude가 tool을 어떻게 호출할지 지정하는 `tool_choice` parameter를 지원합니다. 이 notebook에서는 작동 방식과 사용할 때를 살펴봅니다. 계속하기 전에 Claude tool use의 기본 사항에 익숙한지 확인하세요.

`tool_choice`에는 세 선택지가 있습니다.

* `auto`는 제공한 tool을 호출할지 Claude가 결정하게 합니다.
* `tool`은 항상 특정 tool을 사용하도록 강제할 수 있습니다.
* `any`는 제공한 tool 중 하나를 반드시 사용하도록 하지만 특정 tool을 강제하지는 않습니다.

각 선택지를 자세히 살펴보겠습니다. 먼저 Anthropic SDK를 import합니다:


In [31]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

## Auto

`tool_choice`를 `auto`로 설정하면 model이 tool 사용 여부를 자동 결정합니다. tool을 사용할 때 기본 동작입니다.

보여 주기 위해 모의 web search tool을 제공합니다. Claude에 질문하며 일부는 web search tool 호출이 필요하고 다른 것은 스스로 답할 수 있어야 합니다.

먼저 `web_search` tool을 정의합니다. demo를 단순하게 유지하려고 실제 web 검색은 하지 않는다는 점에 유의하세요:


In [137]:
def web_search(topic):
    print(f"pretending to search the web for {topic}")


web_search_tool = {
    "name": "web_search",
    "description": "A tool to retrieve up to date information on a given topic by searching the web",
    "input_schema": {
        "type": "object",
        "properties": {
            "topic": {"type": "string", "description": "The topic to search the web for"},
        },
        "required": ["topic"],
    },
}

다음으로 user_query를 받아 `web_search_tool`과 함께 Claude에 전달하는 함수를 작성합니다.

`tool_choice`도 `auto`로 설정합니다:

```py
tool_choice={"type": "auto"}
```

전체 함수는 다음과 같습니다:


In [145]:
from datetime import date


def chat_with_web_search(user_query):
    messages = [{"role": "user", "content": user_query}]

    system_prompt = f"""
    Answer as many questions as you can using your existing knowledge.
    Only search the web for queries that you can not confidently answer.
    Today's date is {date.today().strftime("%B %d %Y")}
    If you think a user's question involves something in the future that hasn't happened yet, use the search tool.
    """

    response = client.messages.create(
        system=system_prompt,
        model=MODEL_NAME,
        messages=messages,
        max_tokens=1000,
        tool_choice={"type": "auto"},
        tools=[web_search_tool],
    )
    last_content_block = response.content[-1]
    if last_content_block.type == "text":
        print("Claude did NOT call a tool")
        print(f"Assistant: {last_content_block.text}")
    elif last_content_block.type == "tool_use":
        print("Claude wants to use a tool")
        print(last_content_block)

tool 없이 답할 수 있어야 하는 질문부터 시작하겠습니다:


In [139]:
chat_with_web_search("What color is the sky?")

Claude did NOT call a tool
Assistant: The sky appears blue during the day. This is because the Earth's atmosphere scatters more blue light from the sun than other colors, making the sky look blue.


"하늘은 무슨 색인가요?"라고 물으면 tool을 사용하지 않습니다. 답하기 위해 web search tool을 사용해야 할 질문을 해 보겠습니다:


In [140]:
chat_with_web_search("Who won the 2024 Miami Grand Prix?")

Claude wants to use a tool
ToolUseBlock(id='toolu_staging_018nwaaRebX33pHqoZZXDaSw', input={'topic': '2024 Miami Grand Prix winner'}, name='web_search', type='tool_use')


"2024 Miami Grand Prix에서 누가 우승했나요?"라고 물으면 web search tool을 사용합니다!

예제를 몇 개 더 시도하겠습니다:


In [141]:
# Claude는 이에 tool을 사용할 필요가 없어야 합니다:
chat_with_web_search("Who won the superbowl in 2022?")

Claude did NOT call a tool
Assistant: The Los Angeles Rams won Super Bowl LVI in 2022, defeating the Cincinnati Bengals by a score of 23-20. The game was played on February 13, 2022 at SoFi Stadium in Inglewood, California.


In [144]:
# Claude는 이에 tool을 사용해야 합니다:
chat_with_web_search("Who won the superbowl in 2024?")

Claude wants to use a tool
ToolUseBlock(id='toolu_staging_016XPwcprHAgYJBtN7A3jLhb', input={'topic': '2024 Super Bowl winner'}, name='web_search', type='tool_use')


### Prompt가 중요합니다!

`auto`의 `tool_choice`에서는 자세한 prompt 작성에 시간을 들이는 것이 중요합니다. 종종 Claude는 tool을 지나치게 사용하려 할 수 있습니다. 자세한 prompt는 언제 호출하고 언제 호출하지 않을지 판단하도록 돕습니다. 위 예제에서는 system prompt에 구체적인 지시를 포함했습니다:


```py
 system_prompt=f"""
    Answer as many questions as you can using your existing knowledge.  
    Only search the web for queries that you can not confidently answer.
    Today's date is {date.today().strftime("%B %d %Y")}
    If you think a user's question involves something in the future that hasn't happened yet, use the search tool.
"""
```




## 특정 tool 강제하기

`tool_choice`로 특정 tool 사용을 강제할 수 있습니다. 아래 예제는 간단한 tool 두 개를 정의합니다:
* `print_sentiment_scores` - sentiment analysis 데이터를 포함한 잘 구성된 JSON 출력을 생성하도록 Claude를 "유도하는" tool. 자세한 내용은 [Claude와 tool use로 structured JSON 추출하기](https://github.com/anthropics/anthropic-cookbook/blob/main/tool_use/extracting_structured_json.ipynb)를 참고하세요.
* `calculator` - 숫자 두 개를 받아 더하는 매우 간단한 calculator tool


In [111]:
tools = [
    {
        "name": "print_sentiment_scores",
        "description": "Prints the sentiment scores of a given tweet or piece of text.",
        "input_schema": {
            "type": "object",
            "properties": {
                "positive_score": {
                    "type": "number",
                    "description": "The positive sentiment score, ranging from 0.0 to 1.0.",
                },
                "negative_score": {
                    "type": "number",
                    "description": "The negative sentiment score, ranging from 0.0 to 1.0.",
                },
                "neutral_score": {
                    "type": "number",
                    "description": "The neutral sentiment score, ranging from 0.0 to 1.0.",
                },
            },
            "required": ["positive_score", "negative_score", "neutral_score"],
        },
    },
    {
        "name": "calculator",
        "description": "Adds two number",
        "input_schema": {
            "type": "object",
            "properties": {
                "num1": {"type": "number", "description": "first number to add"},
                "num2": {"type": "number", "description": "second number to add"},
            },
            "required": ["num1", "num2"],
        },
    },
]

목표는 tweet을 받아 기본 sentiment analysis를 출력하는 `analyze_tweet_sentiment` 함수를 작성하는 것입니다. 최종적으로 sentiment analysis tool을 "강제"하지만 먼저 **강제하지 않았을** 때를 보여 줍니다.

첫 "나쁜" version에서는 두 tool을 모두 제공합니다. 비교를 위해 tool_choice를 "auto"로 시작합니다:

```py
tool_choice={"type": "auto"}
```

특정 tool 강제의 영향을 쉽게 보기 위해 의도적으로 잘 작성된 prompt를 제공하지 않는다는 점에 유의하세요.


In [124]:
def analyze_tweet_sentiment(query):
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "auto"},
        messages=[{"role": "user", "content": query}],
    )
    print(response)

"세상에, 방금 정말 놀라운 요리를 만들었어요!"라는 tweet으로 함수를 호출하면 어떻게 되는지 보겠습니다.


In [125]:
analyze_tweet_sentiment("Holy cow, I just made the most incredible meal!")

ToolsBetaMessage(id='msg_staging_01ApgXx7W7qsDugdaRWh6p21', content=[TextBlock(text="That's great to hear! I don't actually have the capability to assess sentiment from text, but it sounds like you're really excited and proud of the incredible meal you made. Cooking something delicious that you're proud of can definitely give a sense of accomplishment and happiness. Well done on creating such an amazing dish!", type='text')], model='claude-sonnet-4-6', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(input_tokens=429, output_tokens=69))


Claude는 sentiment analysis tool을 호출하지 않습니다:
> "정말 좋은 소식이네요! 실제로 text의 감정을 평가할 능력은 없지만, 만든 놀라운 요리에 대해 정말 신나고 자랑스러워하는 것 같아요

다음으로 누군가 이렇게 tweet했다고 상상해 보세요. "고양이들이 정말 좋아요! 네 마리였는데 방금 두 마리 더 입양했어요! 이제 몇 마리인지 맞혀 보세요?"


In [128]:
analyze_tweet_sentiment(
    "I love my cats! I had four and just adopted 2 more! Guess how many I have now?"
)

ToolsBetaMessage(id='msg_staging_018gTrwrx6YwBR2jjhdPooVg', content=[TextBlock(text="That's wonderful that you love your cats and adopted two more! To figure out how many cats you have now, I can use the calculator tool:", type='text'), ToolUseBlock(id='toolu_staging_01RFker5oMQoY6jErz5prmZg', input={'num1': 4, 'num2': 2}, name='calculator', type='tool_use')], model='claude-sonnet-4-6', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=442, output_tokens=101))


Claude는 calculator tool을 호출하려 합니다:

> ToolUseBlock(id='toolu_staging_01RFker5oMQoY6jErz5prmZg', input={'num1': 4, 'num2': 2}, name='calculator', type='tool_use')


현재 구현은 분명히 원하는 일을 하지 않습니다(주로 실패하도록 설정했기 때문입니다).

다음으로 `tool_choice`를 update하여 **항상** `print_sentiment_scores`를 사용하도록 강제합니다:

```py
tool_choice={"type": "tool", "name": "print_sentiment_scores"}
```

`type`을 `tool`로 설정하는 것 외에 특정 tool 이름도 제공해야 합니다.


In [132]:
def analyze_tweet_sentiment(query):
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "tool", "name": "print_sentiment_scores"},
        messages=[{"role": "user", "content": query}],
    )
    print(response)

이제 이전과 같은 prompt로 요청하면 항상 `print_sentiment_scores`를 호출합니다:


In [133]:
analyze_tweet_sentiment("Holy cow, I just made the most incredible meal!")

ToolsBetaMessage(id='msg_staging_018GtYk8Xvee3w8Eeh6pbgoq', content=[ToolUseBlock(id='toolu_staging_01FMRQ9pZniZqFUGQwTcFU4N', input={'positive_score': 0.9, 'negative_score': 0.0, 'neutral_score': 0.1}, name='print_sentiment_scores', type='tool_use')], model='claude-sonnet-4-6', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=527, output_tokens=79))


Claude는 `print_sentiment_scores`를 호출합니다:

> ToolUseBlock(id='toolu_staging_01FMRQ9pZniZqFUGQwTcFU4N', input={'positive_score': 0.9, 'negative_score': 0.0, 'neutral_score': 0.1}, name='print_sentiment_scores', type='tool_use')

"수학적인" tweet으로 혼란시키려 해도 항상 `print_sentiment_scores`를 호출합니다:


In [134]:
analyze_tweet_sentiment(
    "I love my cats! I had four and just adopted 2 more! Guess how many I have now?"
)

ToolsBetaMessage(id='msg_staging_01RACamfrHdpvLxWaNwDfZEF', content=[ToolUseBlock(id='toolu_staging_01Wb6ZKSwKvqVSKLDAte9cKU', input={'positive_score': 0.8, 'negative_score': 0.0, 'neutral_score': 0.2}, name='print_sentiment_scores', type='tool_use')], model='claude-sonnet-4-6', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=540, output_tokens=79))


`print_sentiment_scores` 호출을 강제해도 기본적인 prompt engineering을 적용해야 합니다:


In [135]:
def analyze_tweet_sentiment(query):
    prompt = f"""
    Analyze the sentiment in the following tweet:
    <tweet>{query}</tweet>
    """

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "auto"},
        messages=[{"role": "user", "content": prompt}],
    )
    print(response)

## Any

마지막 `tool_choice` 선택지는 `any`로, "tool을 호출해야 하지만 어느 것인지는 선택할 수 있다"고 알려 줍니다. Claude로 SMS chatbot을 만든다고 상상해 보세요. 실제로 사용자와 "소통"하는 유일한 방법은 SMS text message입니다.

아래 예제는 tool 두 개에 접근하는 매우 간단한 문자 message assistant를 만듭니다:
* `send_text_to_user`는 사용자에게 text message를 보냅니다.
* `get_customer_info`는 username으로 고객 데이터를 조회합니다.

항상 이 tool 중 하나를 호출하고 tool 없는 응답은 하지 않는 chatbot을 만들려는 것입니다. 모든 상황에서 text message를 보내려 하거나 추가 고객 정보를 얻기 위해 `get_customer_info`를 호출해야 합니다.

가장 중요한 것은 `tool_choice`를 "any"로 설정하는 것입니다:

```py
tool_choice={"type": "any"}
```


In [162]:
def send_text_to_user(text):
    # 사용자에게 text 전송
    # 간단히 하기 위해 text만 출력합니다:
    print(f"TEXT MESSAGE SENT: {text}")


def get_customer_info(username):
    return {
        "username": username,
        "email": f"{username}@email.com",
        "purchases": [
            {"id": 1, "product": "computer mouse"},
            {"id": 2, "product": "screen protector"},
            {"id": 3, "product": "usb charging cable"},
        ],
    }


tools = [
    {
        "name": "send_text_to_user",
        "description": "Sends a text message to a user",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "The piece of text to be sent to the user via text message",
                },
            },
            "required": ["text"],
        },
    },
    {
        "name": "get_customer_info",
        "description": "gets information on a customer based on the customer's username.  Response includes email, username, and previous purchases. Only call this tool once a user has provided you with their username",
        "input_schema": {
            "type": "object",
            "properties": {
                "username": {
                    "type": "string",
                    "description": "The username of the user in question. ",
                },
            },
            "required": ["username"],
        },
    },
]

system_prompt = """
All your communication with a user is done via text message.
Only call tools when you have enough information to accurately call them.
Do not call the get_customer_info tool until a user has provided you with their username. This is important.
If you do not know a user's username, simply ask a user for their username.
"""


def sms_chatbot(user_message):
    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        system=system_prompt,
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "any"},
        messages=messages,
    )
    if response.stop_reason == "tool_use":
        last_content_block = response.content[-1]
        if last_content_block.type == "tool_use":
            tool_name = last_content_block.name
            tool_inputs = last_content_block.input
            print(f"=======Claude Wants To Call The {tool_name} Tool=======")
            if tool_name == "send_text_to_user":
                send_text_to_user(tool_inputs["text"])
            elif tool_name == "get_customer_info":
                print(get_customer_info(tool_inputs["username"]))
            else:
                print("Oh dear, that tool doesn't exist!")

    else:
        print("No tool was called. This shouldn't happen!")

간단하게 시작하겠습니다:


In [163]:
sms_chatbot("Hey there! How are you?")

=======Claude Wants To Call The send_text_to_user Tool=======
TEXT MESSAGE SENT: Hello! I'm doing well, thanks for asking. How can I assist you today?


Claude는 `send_text_to_user`를 호출하여 응답합니다.

다음으로 조금 더 까다로운 것을 묻겠습니다:


In [164]:
sms_chatbot("I need help looking up an order")

=======Claude Wants To Call The send_text_to_user Tool=======
TEXT MESSAGE SENT: Hi there, to look up your order details I'll need your username first. Can you please provide me with your username?


Claude는 username을 알려 달라는 text message를 보내려 합니다.

이제 username을 제공하면 어떻게 되는지 보겠습니다:


In [165]:
sms_chatbot("I need help looking up an order.  My username is jenny76")

=======Claude Wants To Call The get_customer_info Tool=======
{'username': 'jenny76', 'email': 'jenny76@email.com', 'purchases': [{'id': 1, 'product': 'computer mouse'}, {'id': 2, 'product': 'screen protector'}, {'id': 3, 'product': 'usb charging cable'}]}


기대한 대로 `get_customer_info`를 호출합니다!

횡설수설하는 message를 보내도 여전히 tool 중 하나를 호출합니다:


In [166]:
sms_chatbot("askdj aksjdh asjkdbhas kjdhas 1+1 ajsdh")

=======Claude Wants To Call The send_text_to_user Tool=======
TEXT MESSAGE SENT: I'm afraid I didn't understand your query. Could you please rephrase what you need help with?
